# Confidence Gap Examples: Genesys < 80% vs Deepgram > 95%

Extracts utterances where Genesys r2d2 STT confidence was below 80%
while Deepgram Nova-3 confidence exceeded 95% on the same audio.
Outputs a markdown document to `analysis_results/`.

In [1]:
from pathlib import Path

import pandas as pd

REPO_ROOT = Path("..").resolve()
EB_CORRELATION_CSV = REPO_ROOT / "analysis_results" / "cross_system_eb" / "eb_correlation.csv"
OUTPUT_DIR = REPO_ROOT / "analysis_results"
OUTPUT_FILE = OUTPUT_DIR / "confidence_gap_examples.md"

GENESYS_THRESHOLD = 0.80
DEEPGRAM_THRESHOLD = 0.95
MAX_EXAMPLES = 25

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Source: {EB_CORRELATION_CSV}")
print(f"Output: {OUTPUT_FILE}")

Source: /Users/xnxn040/PycharmProjects/notifications-spike/analysis_results/cross_system_eb/eb_correlation.csv
Output: /Users/xnxn040/PycharmProjects/notifications-spike/analysis_results/confidence_gap_examples.md


In [2]:
df = pd.read_csv(EB_CORRELATION_CSV)

# Filter out negative-latency false matches
df = df[df["true_latency_ms"] >= 0].copy()

print(f"Total matched utterance pairs: {len(df)}")
print(f"Genesys confidence range: {df['genesys_confidence'].min():.1%} – {df['genesys_confidence'].max():.1%}")
print(f"Deepgram confidence range: {df['deepgram_confidence'].min():.1%} – {df['deepgram_confidence'].max():.1%}")

Total matched utterance pairs: 62
Genesys confidence range: 59.5% – 95.5%
Deepgram confidence range: 55.8% – 100.0%


In [3]:
gap_df = df[
    (df["genesys_confidence"] < GENESYS_THRESHOLD)
    & (df["deepgram_confidence"] > DEEPGRAM_THRESHOLD)
].copy()

gap_df["confidence_gap"] = gap_df["deepgram_confidence"] - gap_df["genesys_confidence"]
gap_df = gap_df.sort_values("confidence_gap", ascending=False)

examples = gap_df.head(MAX_EXAMPLES)

print(f"Utterances matching criteria: {len(gap_df)}")
print(f"Examples to include: {len(examples)}")
print(f"Mean confidence gap: {gap_df['confidence_gap'].mean():.1%}")

Utterances matching criteria: 19
Examples to include: 19
Mean confidence gap: 26.2%


In [4]:
examples[
    ["deepgram_transcript", "genesys_transcript", "deepgram_confidence", "genesys_confidence", "confidence_gap", "similarity"]
].reset_index(drop=True)

,deepgram_transcript,genesys_transcript,deepgram_confidence,genesys_confidence,confidence_gap,similarity
0,This old man is all that's left.,this old man is all this left,0.984,0.611,0.373,0.949
1,And the cynical,the cynical,0.972,0.603,0.369,0.846
2,go along with them on the assumption.,go along with them on the assumption,0.988,0.656,0.332,1.000
3,Truth is,true dish,0.993,0.671,0.322,0.706
4,"You hear me, you fucking faggot?",hear me you fucking bad,0.998,0.676,0.322,0.792
5,It takes grass balls,takes grass balls,0.998,0.711,0.287,0.919
6,All Negro men are not to be trusted around our...,oh nibra bend that to be trusted around our wi...,0.986,0.699,0.287,0.725
7,"Royalty, nobility, the gentry,",nobility the gentry and,0.968,0.694,0.274,0.760
8,I don't get this yet.,i don't get a,0.969,0.712,0.257,0.710
9,Who has had the unmitigated terminology,who has had the unheated ten years,0.959,0.707,0.252,0.712


In [5]:
lines = [
    "# Confidence Gap Examples: Genesys < 80% vs Deepgram > 95%\n",
    "",
    f"**Criteria:** Genesys r2d2 confidence < {GENESYS_THRESHOLD:.0%} AND Deepgram Nova-3 confidence > {DEEPGRAM_THRESHOLD:.0%}\n",
    f"**Source:** `cross_system_eb/eb_correlation.csv` ({len(df)} matched utterance pairs)\n",
    f"**Matches found:** {len(gap_df)} | **Shown:** {len(examples)}\n",
    f"**Mean confidence gap:** {gap_df['confidence_gap'].mean():.1%}\n",
    "",
    "---\n",
    "",
]

for i, (_, row) in enumerate(examples.iterrows(), start=1):
    lines.append(f"## Example {i}\n")
    lines.append("")
    lines.append(f"| Metric | Value |")
    lines.append(f"| --- | --- |")
    lines.append(f"| **Deepgram transcript** | {row['deepgram_transcript']} |")
    lines.append(f"| **Genesys transcript** | {row['genesys_transcript']} |")
    lines.append(f"| **Deepgram confidence** | {row['deepgram_confidence']:.1%} |")
    lines.append(f"| **Genesys confidence** | {row['genesys_confidence']:.1%} |")
    lines.append(f"| **Confidence gap** | {row['confidence_gap']:.1%} |")
    lines.append(f"| **Text similarity** | {row['similarity']:.1%} |")
    lines.append(f"| **Channel** | {row['channel']} |")
    lines.append("")

OUTPUT_FILE.write_text("\n".join(lines))
print(f"Written {len(examples)} examples to {OUTPUT_FILE}")

Written 19 examples to /Users/xnxn040/PycharmProjects/notifications-spike/analysis_results/confidence_gap_examples.md
